In [1]:
!pip install pyspark

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName('TelecomAnalytics') \
    .getOrCreate()

print('Spark version:', spark.version)

Spark version: 4.0.2


In [3]:
from google.colab import files
uploaded = files.upload()

Saving customers.csv to customers (1).csv
Saving payments.csv to payments (1).csv
Saving plans.json to plans (1).json
Saving usage.csv to usage (1).csv


In [4]:
customers_df = spark.read.csv('customers.csv', header=True, inferSchema=True)
customers_df.show()

+-----------+-------------+---------+-----------+---+------+-------+--------+
|customer_id|customer_name|     city|      state|age|gender|plan_id|  status|
+-----------+-------------+---------+-----------+---+------+-------+--------+
|        101| Rahul Sharma|Hyderabad|  Telangana| 35|  Male|   P101|  Active|
|        102|  Priya Reddy|Bangalore|  Karnataka| 29|Female|   P102|  Active|
|        103|   Amit Kumar|   Mumbai|Maharashtra| 42|  Male|   P103|Inactive|
|        104|  Sneha Patel|  Chennai| Tamil Nadu| 31|Female|   P101|  Active|
|        105|   Farhan Ali|    Delhi|      Delhi| 55|  Male|   P104|  Active|
|        106|   Neha Singh|     Pune|Maharashtra| 38|Female|   P102|  Active|
|        107|  Arjun Verma|Hyderabad|  Telangana| 26|  Male|   P103|Inactive|
|        108|   Meera Nair|    Kochi|     Kerala| 48|Female|   P104|  Active|
|        109|    Kiran Rao|Bangalore|  Karnataka| 33|  Male|   P101|  Active|
|        110|  Nisha Reddy|    Delhi|      Delhi| 41|Female|   P

In [5]:
usage_df = spark.read.csv('usage.csv', header=True, inferSchema=True)
usage_df.show()

+--------+-----------+-------------------+------------+------------+---------+
|usage_id|customer_id|        usage_month|data_used_gb|call_minutes|sms_count|
+--------+-----------+-------------------+------------+------------+---------+
|    1001|        101|2026-01-01 00:00:00|          45|         900|      120|
|    1002|        102|2026-01-01 00:00:00|          30|         600|       80|
|    1003|        103|2026-01-01 00:00:00|          12|         250|       40|
|    1004|        104|2026-01-01 00:00:00|          55|        1100|      150|
|    1005|        105|2026-01-01 00:00:00|          75|        1500|      200|
|    1006|        106|2026-01-01 00:00:00|          28|         500|       60|
|    1007|        107|2026-01-01 00:00:00|          10|         200|       20|
|    1008|        108|2026-01-01 00:00:00|          80|        1600|      250|
|    1009|        109|2026-01-01 00:00:00|          48|         950|      100|
|    1010|        110|2026-01-01 00:00:00|          

In [6]:
payments_df = spark.read.csv('payments.csv', header=True, inferSchema=True)
payments_df.show()

+----------+-----------+-------------------+-----------+------------+--------------+
|payment_id|customer_id|         bill_month|amount_paid|payment_mode|payment_status|
+----------+-----------+-------------------+-----------+------------+--------------+
|      5001|        101|2026-01-01 00:00:00|        499|         UPI|       Success|
|      5002|        102|2026-01-01 00:00:00|        799|        Card|       Success|
|      5003|        103|2026-01-01 00:00:00|        299|        Cash|        Failed|
|      5004|        104|2026-01-01 00:00:00|        499|         UPI|       Success|
|      5005|        105|2026-01-01 00:00:00|       1199|        Card|       Success|
|      5006|        106|2026-01-01 00:00:00|        799|         UPI|       Success|
|      5007|        107|2026-01-01 00:00:00|        299|        Cash|       Pending|
|      5008|        108|2026-01-01 00:00:00|       1199|        Card|       Success|
|      5009|        109|2026-01-01 00:00:00|        499|         

In [7]:
plans_raw_df = spark.read.option('multiLine', True).json('plans.json')
plans_raw_df.show(truncate=False)

+-------------+---------------------------+-----------+-------+------------+
|data_limit_gb|features                   |monthly_fee|plan_id|plan_name   |
+-------------+---------------------------+-----------+-------+------------+
|50           |{false, National, true}    |499        |P101   |Smart Basic |
|75           |{true, National, true}     |799        |P102   |Smart Plus  |
|25           |{false, NULL, false}       |299        |P103   |Budget Saver|
|100          |{true, International, true}|1199       |P104   |Premium Max |
+-------------+---------------------------+-----------+-------+------------+



In [8]:
customers_df.printSchema()
usage_df.printSchema()
payments_df.printSchema()
plans_raw_df.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- plan_id: string (nullable = true)
 |-- status: string (nullable = true)

root
 |-- usage_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- usage_month: timestamp (nullable = true)
 |-- data_used_gb: integer (nullable = true)
 |-- call_minutes: integer (nullable = true)
 |-- sms_count: integer (nullable = true)

root
 |-- payment_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- bill_month: timestamp (nullable = true)
 |-- amount_paid: integer (nullable = true)
 |-- payment_mode: string (nullable = true)
 |-- payment_status: string (nullable = true)

root
 |-- data_limit_gb: long (nullable = true)
 |-- features: struct (nullable = true)
 |    |-- ott_included: boolean (nullable = true)
 |

In [9]:
print('customers count :', customers_df.count())
print('usage count     :', usage_df.count())
print('payments count  :', payments_df.count())
print('plans count     :', plans_raw_df.count())

customers count : 12
usage count     : 15
payments count  : 15
plans count     : 4


In [10]:
customers_df.write.mode('overwrite').parquet('bronze/customers')
usage_df.write.mode('overwrite').parquet('bronze/usage')
payments_df.write.mode('overwrite').parquet('bronze/payments')
plans_raw_df.write.mode('overwrite').parquet('bronze/plans')

In [11]:
customers_df.filter(F.col('plan_id').isNull()).show()

+-----------+-------------+---------+---------+---+------+-------+------+
|customer_id|customer_name|     city|    state|age|gender|plan_id|status|
+-----------+-------------+---------+---------+---+------+-------+------+
|        112|  Ayesha Khan|Hyderabad|Telangana| 28|Female|   NULL|Active|
+-----------+-------------+---------+---------+---+------+-------+------+



In [12]:
usage_df.filter(F.col('data_used_gb').isNull()).show()

+--------+-----------+-------------------+------------+------------+---------+
|usage_id|customer_id|        usage_month|data_used_gb|call_minutes|sms_count|
+--------+-----------+-------------------+------------+------------+---------+
|    1015|        105|2026-02-01 00:00:00|        NULL|        1450|      210|
+--------+-----------+-------------------+------------+------------+---------+



In [13]:
payments_df.filter(F.col('amount_paid').isNull()).show()

+----------+-----------+-------------------+-----------+------------+--------------+
|payment_id|customer_id|         bill_month|amount_paid|payment_mode|payment_status|
+----------+-----------+-------------------+-----------+------------+--------------+
|      5011|        112|2026-01-01 00:00:00|       NULL|         UPI|       Success|
+----------+-----------+-------------------+-----------+------------+--------------+



In [14]:
payments_df.filter(F.col('payment_mode').isNull()).show()

+----------+-----------+-------------------+-----------+------------+--------------+
|payment_id|customer_id|         bill_month|amount_paid|payment_mode|payment_status|
+----------+-----------+-------------------+-----------+------------+--------------+
|      5015|        105|2026-02-01 00:00:00|       1199|        NULL|       Pending|
+----------+-----------+-------------------+-----------+------------+--------------+



In [15]:
usage_clean = usage_df.fillna({'data_used_gb': 0})
usage_clean.show()

+--------+-----------+-------------------+------------+------------+---------+
|usage_id|customer_id|        usage_month|data_used_gb|call_minutes|sms_count|
+--------+-----------+-------------------+------------+------------+---------+
|    1001|        101|2026-01-01 00:00:00|          45|         900|      120|
|    1002|        102|2026-01-01 00:00:00|          30|         600|       80|
|    1003|        103|2026-01-01 00:00:00|          12|         250|       40|
|    1004|        104|2026-01-01 00:00:00|          55|        1100|      150|
|    1005|        105|2026-01-01 00:00:00|          75|        1500|      200|
|    1006|        106|2026-01-01 00:00:00|          28|         500|       60|
|    1007|        107|2026-01-01 00:00:00|          10|         200|       20|
|    1008|        108|2026-01-01 00:00:00|          80|        1600|      250|
|    1009|        109|2026-01-01 00:00:00|          48|         950|      100|
|    1010|        110|2026-01-01 00:00:00|          

In [16]:
payments_clean = payments_df.fillna({'amount_paid': 0})
payments_clean.show()

+----------+-----------+-------------------+-----------+------------+--------------+
|payment_id|customer_id|         bill_month|amount_paid|payment_mode|payment_status|
+----------+-----------+-------------------+-----------+------------+--------------+
|      5001|        101|2026-01-01 00:00:00|        499|         UPI|       Success|
|      5002|        102|2026-01-01 00:00:00|        799|        Card|       Success|
|      5003|        103|2026-01-01 00:00:00|        299|        Cash|        Failed|
|      5004|        104|2026-01-01 00:00:00|        499|         UPI|       Success|
|      5005|        105|2026-01-01 00:00:00|       1199|        Card|       Success|
|      5006|        106|2026-01-01 00:00:00|        799|         UPI|       Success|
|      5007|        107|2026-01-01 00:00:00|        299|        Cash|       Pending|
|      5008|        108|2026-01-01 00:00:00|       1199|        Card|       Success|
|      5009|        109|2026-01-01 00:00:00|        499|         

In [17]:
payments_clean = payments_clean.fillna({'payment_mode': 'Not Provided'})
payments_clean.show()

+----------+-----------+-------------------+-----------+------------+--------------+
|payment_id|customer_id|         bill_month|amount_paid|payment_mode|payment_status|
+----------+-----------+-------------------+-----------+------------+--------------+
|      5001|        101|2026-01-01 00:00:00|        499|         UPI|       Success|
|      5002|        102|2026-01-01 00:00:00|        799|        Card|       Success|
|      5003|        103|2026-01-01 00:00:00|        299|        Cash|        Failed|
|      5004|        104|2026-01-01 00:00:00|        499|         UPI|       Success|
|      5005|        105|2026-01-01 00:00:00|       1199|        Card|       Success|
|      5006|        106|2026-01-01 00:00:00|        799|         UPI|       Success|
|      5007|        107|2026-01-01 00:00:00|        299|        Cash|       Pending|
|      5008|        108|2026-01-01 00:00:00|       1199|        Card|       Success|
|      5009|        109|2026-01-01 00:00:00|        499|         

In [18]:
customers_clean = customers_df.fillna({'plan_id': 'UNKNOWN'})
customers_clean.show()

+-----------+-------------+---------+-----------+---+------+-------+--------+
|customer_id|customer_name|     city|      state|age|gender|plan_id|  status|
+-----------+-------------+---------+-----------+---+------+-------+--------+
|        101| Rahul Sharma|Hyderabad|  Telangana| 35|  Male|   P101|  Active|
|        102|  Priya Reddy|Bangalore|  Karnataka| 29|Female|   P102|  Active|
|        103|   Amit Kumar|   Mumbai|Maharashtra| 42|  Male|   P103|Inactive|
|        104|  Sneha Patel|  Chennai| Tamil Nadu| 31|Female|   P101|  Active|
|        105|   Farhan Ali|    Delhi|      Delhi| 55|  Male|   P104|  Active|
|        106|   Neha Singh|     Pune|Maharashtra| 38|Female|   P102|  Active|
|        107|  Arjun Verma|Hyderabad|  Telangana| 26|  Male|   P103|Inactive|
|        108|   Meera Nair|    Kochi|     Kerala| 48|Female|   P104|  Active|
|        109|    Kiran Rao|Bangalore|  Karnataka| 33|  Male|   P101|  Active|
|        110|  Nisha Reddy|    Delhi|      Delhi| 41|Female|   P

In [19]:
customers_clean = customers_clean.withColumn(
      'data_quality_status',
      F.when(F.col('plan_id') == 'UNKNOWN', 'Incomplete').otherwise('Complete')
      )

usage_clean = usage_clean.withColumn(
       'data_quality_status',
         F.when(F.col('data_used_gb') == 0, 'Incomplete').otherwise('Complete')
)

payments_clean = payments_clean.withColumn(
          'data_quality_status',
            F.when(
            (F.col('amount_paid') == 0) | (F.col('payment_mode') == 'Not Provided'),
            'Incomplete'
          ).otherwise('Complete'))
print('=== customers data_quality_status ===')
customers_clean.select('customer_id', 'customer_name', 'plan_id', 'data_quality_status').show()

print('=== usage data_quality_status ===')
usage_clean.select('usage_id', 'customer_id', 'data_used_gb', 'data_quality_status').show()
print('=== payments data_quality_status ===')
payments_clean.select('payment_id', 'customer_id', 'amount_paid', 'payment_mode', 'data_quality_status').show()


=== customers data_quality_status ===
+-----------+-------------+-------+-------------------+
|customer_id|customer_name|plan_id|data_quality_status|
+-----------+-------------+-------+-------------------+
|        101| Rahul Sharma|   P101|           Complete|
|        102|  Priya Reddy|   P102|           Complete|
|        103|   Amit Kumar|   P103|           Complete|
|        104|  Sneha Patel|   P101|           Complete|
|        105|   Farhan Ali|   P104|           Complete|
|        106|   Neha Singh|   P102|           Complete|
|        107|  Arjun Verma|   P103|           Complete|
|        108|   Meera Nair|   P104|           Complete|
|        109|    Kiran Rao|   P101|           Complete|
|        110|  Nisha Reddy|   P102|           Complete|
|        111|   Ravi Kumar|   P105|           Complete|
|        112|  Ayesha Khan|UNKNOWN|         Incomplete|
+-----------+-------------+-------+-------------------+

=== usage data_quality_status ===
+--------+-----------+---------

In [20]:
customers_clean.write.mode('overwrite').parquet('silver/customers')
usage_clean.write.mode('overwrite').parquet('silver/usage')
payments_clean.write.mode('overwrite').parquet('silver/payments')

In [21]:
plans_flat = plans_raw_df.select(
      'plan_id',
      'plan_name',
      'monthly_fee',
      'data_limit_gb',
       F.col('features.unlimited_calls').alias('unlimited_calls'),
       F.col('features.ott_included').alias('ott_included'),
       F.col('features.roaming').alias('roaming'))
plans_flat.show(truncate=False)


+-------+------------+-----------+-------------+---------------+------------+-------------+
|plan_id|plan_name   |monthly_fee|data_limit_gb|unlimited_calls|ott_included|roaming      |
+-------+------------+-----------+-------------+---------------+------------+-------------+
|P101   |Smart Basic |499        |50           |true           |false       |National     |
|P102   |Smart Plus  |799        |75           |true           |true        |National     |
|P103   |Budget Saver|299        |25           |false          |false       |NULL         |
|P104   |Premium Max |1199       |100          |true           |true        |International|
+-------+------------+-----------+-------------+---------------+------------+-------------+



In [22]:
plans_flat.select('plan_id', 'plan_name', 'unlimited_calls').show()

+-------+------------+---------------+
|plan_id|   plan_name|unlimited_calls|
+-------+------------+---------------+
|   P101| Smart Basic|           true|
|   P102|  Smart Plus|           true|
|   P103|Budget Saver|          false|
|   P104| Premium Max|           true|
+-------+------------+---------------+



In [23]:
plans_flat.select('plan_id', 'plan_name', 'unlimited_calls').show()

+-------+------------+---------------+
|plan_id|   plan_name|unlimited_calls|
+-------+------------+---------------+
|   P101| Smart Basic|           true|
|   P102|  Smart Plus|           true|
|   P103|Budget Saver|          false|
|   P104| Premium Max|           true|
+-------+------------+---------------+



In [24]:
plans_flat.select('plan_id', 'plan_name', 'ott_included').show()

+-------+------------+------------+
|plan_id|   plan_name|ott_included|
+-------+------------+------------+
|   P101| Smart Basic|       false|
|   P102|  Smart Plus|        true|
|   P103|Budget Saver|       false|
|   P104| Premium Max|        true|
+-------+------------+------------+



In [25]:
plans_flat.select('plan_id', 'plan_name', 'roaming').show()

+-------+------------+-------------+
|plan_id|   plan_name|      roaming|
+-------+------------+-------------+
|   P101| Smart Basic|     National|
|   P102|  Smart Plus|     National|
|   P103|Budget Saver|         NULL|
|   P104| Premium Max|International|
+-------+------------+-------------+



In [26]:
plans_flat = plans_flat.fillna({'roaming': 'Not Available'})
plans_flat.select('plan_id', 'plan_name', 'roaming').show()

+-------+------------+-------------+
|plan_id|   plan_name|      roaming|
+-------+------------+-------------+
|   P101| Smart Basic|     National|
|   P102|  Smart Plus|     National|
|   P103|Budget Saver|Not Available|
|   P104| Premium Max|International|
+-------+------------+-------------+



In [27]:
plans_flat.write.mode('overwrite').parquet('silver/plans')

In [28]:
cust_plan = customers_clean.join(plans_flat, on='plan_id', how='left')
cust_plan.show(truncate=False)

+-------+-----------+-------------+---------+-----------+---+------+--------+-------------------+------------+-----------+-------------+---------------+------------+-------------+
|plan_id|customer_id|customer_name|city     |state      |age|gender|status  |data_quality_status|plan_name   |monthly_fee|data_limit_gb|unlimited_calls|ott_included|roaming      |
+-------+-----------+-------------+---------+-----------+---+------+--------+-------------------+------------+-----------+-------------+---------------+------------+-------------+
|P101   |101        |Rahul Sharma |Hyderabad|Telangana  |35 |Male  |Active  |Complete           |Smart Basic |499        |50           |true           |false       |National     |
|P102   |102        |Priya Reddy  |Bangalore|Karnataka  |29 |Female|Active  |Complete           |Smart Plus  |799        |75           |true           |true        |National     |
|P103   |103        |Amit Kumar   |Mumbai   |Maharashtra|42 |Male  |Inactive|Complete           |Bud

In [29]:
cust_usage = customers_clean.join(usage_clean, on='customer_id', how='left')
cust_usage.show()

+-----------+-------------+---------+-----------+---+------+-------+--------+-------------------+--------+-------------------+------------+------------+---------+-------------------+
|customer_id|customer_name|     city|      state|age|gender|plan_id|  status|data_quality_status|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|data_quality_status|
+-----------+-------------+---------+-----------+---+------+-------+--------+-------------------+--------+-------------------+------------+------------+---------+-------------------+
|        101| Rahul Sharma|Hyderabad|  Telangana| 35|  Male|   P101|  Active|           Complete|    1012|2026-02-01 00:00:00|          50|        1000|      130|           Complete|
|        101| Rahul Sharma|Hyderabad|  Telangana| 35|  Male|   P101|  Active|           Complete|    1001|2026-01-01 00:00:00|          45|         900|      120|           Complete|
|        102|  Priya Reddy|Bangalore|  Karnataka| 29|Female|   P102|  Active|        

In [30]:
cust_pay = customers_clean.join(payments_clean, on='customer_id', how='left')
cust_pay.show()

+-----------+-------------+---------+-----------+---+------+-------+--------+-------------------+----------+-------------------+-----------+------------+--------------+-------------------+
|customer_id|customer_name|     city|      state|age|gender|plan_id|  status|data_quality_status|payment_id|         bill_month|amount_paid|payment_mode|payment_status|data_quality_status|
+-----------+-------------+---------+-----------+---+------+-------+--------+-------------------+----------+-------------------+-----------+------------+--------------+-------------------+
|        101| Rahul Sharma|Hyderabad|  Telangana| 35|  Male|   P101|  Active|           Complete|      5012|2026-02-01 00:00:00|        499|        Card|       Success|           Complete|
|        101| Rahul Sharma|Hyderabad|  Telangana| 35|  Male|   P101|  Active|           Complete|      5001|2026-01-01 00:00:00|        499|         UPI|       Success|           Complete|
|        102|  Priya Reddy|Bangalore|  Karnataka| 29|Fe

In [31]:
complete_df = customers_clean \
    .join(usage_clean,    on='customer_id', how='left') \
    .join(payments_clean, on='customer_id', how='left') \
    .join(plans_flat,     on='plan_id',     how='left')
complete_df.show(truncate=False)

+-------+-----------+-------------+---------+-----------+---+------+--------+-------------------+--------+-------------------+------------+------------+---------+-------------------+----------+-------------------+-----------+------------+--------------+-------------------+------------+-----------+-------------+---------------+------------+-------------+
|plan_id|customer_id|customer_name|city     |state      |age|gender|status  |data_quality_status|usage_id|usage_month        |data_used_gb|call_minutes|sms_count|data_quality_status|payment_id|bill_month         |amount_paid|payment_mode|payment_status|data_quality_status|plan_name   |monthly_fee|data_limit_gb|unlimited_calls|ott_included|roaming      |
+-------+-----------+-------------+---------+-----------+---+------+--------+-------------------+--------+-------------------+------------+------------+---------+-------------------+----------+-------------------+-----------+------------+--------------+-------------------+------------+--

In [32]:
valid_plans = [r['plan_id'] for r in plans_flat.select('plan_id').collect()]
print('Valid plan IDs:', valid_plans)
customers_clean.filter(
    (F.col('plan_id') == 'UNKNOWN') | (~F.col('plan_id').isin(valid_plans))
    ).show()

Valid plan IDs: ['P101', 'P102', 'P103', 'P104']
+-----------+-------------+---------+-----------+---+------+-------+------+-------------------+
|customer_id|customer_name|     city|      state|age|gender|plan_id|status|data_quality_status|
+-----------+-------------+---------+-----------+---+------+-------+------+-------------------+
|        111|   Ravi Kumar|   Mumbai|Maharashtra| 45|  Male|   P105|Active|           Complete|
|        112|  Ayesha Khan|Hyderabad|  Telangana| 28|Female|UNKNOWN|Active|         Incomplete|
+-----------+-------------+---------+-----------+---+------+-------+------+-------------------+



In [33]:
usage_clean.join(customers_clean, on='customer_id', how='left_anti').show()

+-----------+--------+-------------------+------------+------------+---------+-------------------+
|customer_id|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|data_quality_status|
+-----------+--------+-------------------+------------+------------+---------+-------------------+
|        120|    1011|2026-01-01 00:00:00|          60|        1300|      140|           Complete|
+-----------+--------+-------------------+------------+------------+---------+-------------------+



In [34]:
payments_clean.join(customers_clean, on='customer_id', how='left_anti').show()

+-----------+----------+----------+-----------+------------+--------------+-------------------+
|customer_id|payment_id|bill_month|amount_paid|payment_mode|payment_status|data_quality_status|
+-----------+----------+----------+-----------+------------+--------------+-------------------+
+-----------+----------+----------+-----------+------------+--------------+-------------------+



In [35]:
complete_df = complete_df.withColumn(
      'usage_category',
          F.when(F.col('data_used_gb') >= 70, 'Heavy User')
          .when(F.col('data_used_gb') >= 30, 'Medium User')
          .otherwise('Low User'))
complete_df.select('customer_id', 'customer_name', 'data_used_gb', 'usage_category').show()

+-----------+-------------+------------+--------------+
|customer_id|customer_name|data_used_gb|usage_category|
+-----------+-------------+------------+--------------+
|        101| Rahul Sharma|          50|   Medium User|
|        101| Rahul Sharma|          50|   Medium User|
|        101| Rahul Sharma|          45|   Medium User|
|        101| Rahul Sharma|          45|   Medium User|
|        102|  Priya Reddy|          34|   Medium User|
|        102|  Priya Reddy|          34|   Medium User|
|        102|  Priya Reddy|          30|   Medium User|
|        102|  Priya Reddy|          30|   Medium User|
|        103|   Amit Kumar|          12|      Low User|
|        104|  Sneha Patel|          58|   Medium User|
|        104|  Sneha Patel|          58|   Medium User|
|        104|  Sneha Patel|          55|   Medium User|
|        104|  Sneha Patel|          55|   Medium User|
|        105|   Farhan Ali|           0|      Low User|
|        105|   Farhan Ali|           0|      Lo

In [36]:
complete_df = complete_df.withColumn(
      'payment_category',
        F.when(F.col('amount_paid') >= 1000, 'High Payment')
        .when(F.col('amount_paid') >= 500, 'Medium Payment')
        .otherwise('Low Payment'))
complete_df.select('customer_id', 'customer_name', 'amount_paid', 'payment_category').show()


+-----------+-------------+-----------+----------------+
|customer_id|customer_name|amount_paid|payment_category|
+-----------+-------------+-----------+----------------+
|        101| Rahul Sharma|        499|     Low Payment|
|        101| Rahul Sharma|        499|     Low Payment|
|        101| Rahul Sharma|        499|     Low Payment|
|        101| Rahul Sharma|        499|     Low Payment|
|        102|  Priya Reddy|        799|  Medium Payment|
|        102|  Priya Reddy|        799|  Medium Payment|
|        102|  Priya Reddy|        799|  Medium Payment|
|        102|  Priya Reddy|        799|  Medium Payment|
|        103|   Amit Kumar|        299|     Low Payment|
|        104|  Sneha Patel|        499|     Low Payment|
|        104|  Sneha Patel|        499|     Low Payment|
|        104|  Sneha Patel|        499|     Low Payment|
|        104|  Sneha Patel|        499|     Low Payment|
|        105|   Farhan Ali|       1199|    High Payment|
|        105|   Farhan Ali|    

In [37]:
complete_df = complete_df.withColumn(
      'churn_risk',
          F.when((F.col('status') == 'Inactive') | (F.col('payment_status') != 'Success'),
          'High Risk').when(
          F.col('data_used_gb') < 15, 'Medium Risk').otherwise('Low Risk'))
complete_df.select('customer_id', 'customer_name', 'status', 'payment_status', 'data_used_gb', 'churn_risk').show()

+-----------+-------------+--------+--------------+------------+-----------+
|customer_id|customer_name|  status|payment_status|data_used_gb| churn_risk|
+-----------+-------------+--------+--------------+------------+-----------+
|        101| Rahul Sharma|  Active|       Success|          50|   Low Risk|
|        101| Rahul Sharma|  Active|       Success|          50|   Low Risk|
|        101| Rahul Sharma|  Active|       Success|          45|   Low Risk|
|        101| Rahul Sharma|  Active|       Success|          45|   Low Risk|
|        102|  Priya Reddy|  Active|       Success|          34|   Low Risk|
|        102|  Priya Reddy|  Active|       Success|          34|   Low Risk|
|        102|  Priya Reddy|  Active|       Success|          30|   Low Risk|
|        102|  Priya Reddy|  Active|       Success|          30|   Low Risk|
|        103|   Amit Kumar|Inactive|        Failed|          12|  High Risk|
|        104|  Sneha Patel|  Active|       Success|          58|   Low Risk|

In [38]:
complete_df = complete_df.withColumn(
      'over_usage_gb',
          F.col('data_used_gb') - F.col('data_limit_gb')
          )
complete_df.select('customer_id', 'customer_name', 'data_used_gb', 'data_limit_gb', 'over_usage_gb').show()

+-----------+-------------+------------+-------------+-------------+
|customer_id|customer_name|data_used_gb|data_limit_gb|over_usage_gb|
+-----------+-------------+------------+-------------+-------------+
|        101| Rahul Sharma|          50|           50|            0|
|        101| Rahul Sharma|          50|           50|            0|
|        101| Rahul Sharma|          45|           50|           -5|
|        101| Rahul Sharma|          45|           50|           -5|
|        102|  Priya Reddy|          34|           75|          -41|
|        102|  Priya Reddy|          34|           75|          -41|
|        102|  Priya Reddy|          30|           75|          -45|
|        102|  Priya Reddy|          30|           75|          -45|
|        103|   Amit Kumar|          12|           25|          -13|
|        104|  Sneha Patel|          58|           50|            8|
|        104|  Sneha Patel|          58|           50|            8|
|        104|  Sneha Patel|       

In [39]:
complete_df = complete_df.withColumn(
      'over_usage_flag',
          F.when(F.col('over_usage_gb') > 0, 'Yes').otherwise('No')
          )
complete_df.select('customer_id', 'customer_name', 'over_usage_gb', 'over_usage_flag').show()

+-----------+-------------+-------------+---------------+
|customer_id|customer_name|over_usage_gb|over_usage_flag|
+-----------+-------------+-------------+---------------+
|        101| Rahul Sharma|            0|             No|
|        101| Rahul Sharma|            0|             No|
|        101| Rahul Sharma|           -5|             No|
|        101| Rahul Sharma|           -5|             No|
|        102|  Priya Reddy|          -41|             No|
|        102|  Priya Reddy|          -41|             No|
|        102|  Priya Reddy|          -45|             No|
|        102|  Priya Reddy|          -45|             No|
|        103|   Amit Kumar|          -13|             No|
|        104|  Sneha Patel|            8|            Yes|
|        104|  Sneha Patel|            8|            Yes|
|        104|  Sneha Patel|            5|            Yes|
|        104|  Sneha Patel|            5|            Yes|
|        105|   Farhan Ali|         -100|             No|
|        105| 

In [40]:
customers_clean.groupBy('city').count().orderBy('count', ascending=False).show()

+---------+-----+
|     city|count|
+---------+-----+
|Hyderabad|    3|
|Bangalore|    2|
|   Mumbai|    2|
|    Delhi|    2|
|    Kochi|    1|
|  Chennai|    1|
|     Pune|    1|
+---------+-----+



In [41]:
customers_clean.groupBy('state').count().orderBy('count', ascending=False).show()

+-----------+-----+
|      state|count|
+-----------+-----+
|  Telangana|    3|
|Maharashtra|    3|
|  Karnataka|    2|
|      Delhi|    2|
|     Kerala|    1|
| Tamil Nadu|    1|
+-----------+-----+



In [42]:
customers_clean.groupBy('plan_id').count().orderBy('count', ascending=False).show()

+-------+-----+
|plan_id|count|
+-------+-----+
|   P102|    3|
|   P101|    3|
|   P103|    2|
|   P104|    2|
|   P105|    1|
|UNKNOWN|    1|
+-------+-----+



In [43]:
complete_df.groupBy('usage_category').agg(
      F.countDistinct('customer_id').alias('customer_count')
      ).show()

+--------------+--------------+
|usage_category|customer_count|
+--------------+--------------+
|   Medium User|             5|
|    Heavy User|             2|
|      Low User|             6|
+--------------+--------------+



In [44]:
complete_df.groupBy('churn_risk').agg(
      F.countDistinct('customer_id').alias('customer_count')
      ).show()

+-----------+--------------+
| churn_risk|customer_count|
+-----------+--------------+
|   Low Risk|            10|
|Medium Risk|             1|
|  High Risk|             3|
+-----------+--------------+



In [45]:
complete_df.groupBy('plan_id', 'plan_name').agg(
      F.sum('data_used_gb').alias('total_data_gb')
      ).orderBy('total_data_gb', ascending=False).show()

+-------+------------+-------------+
|plan_id|   plan_name|total_data_gb|
+-------+------------+-------------+
|   P101| Smart Basic|          464|
|   P104| Premium Max|          230|
|   P102|  Smart Plus|          188|
|   P103|Budget Saver|           22|
|   P105|        NULL|         NULL|
|UNKNOWN|        NULL|         NULL|
+-------+------------+-------------+



In [46]:
complete_df.groupBy('plan_id', 'plan_name').agg(
      F.round(F.avg('data_used_gb'), 2).alias('avg_data_gb')
      ).show()

+-------+------------+-----------+
|plan_id|   plan_name|avg_data_gb|
+-------+------------+-----------+
|   P105|        NULL|       NULL|
|   P103|Budget Saver|       11.0|
|   P101| Smart Basic|      51.56|
|   P104| Premium Max|       46.0|
|UNKNOWN|        NULL|       NULL|
|   P102|  Smart Plus|      31.33|
+-------+------------+-----------+



In [47]:
complete_df.groupBy('city').agg(
      F.sum('call_minutes').alias('total_call_minutes')
      ).orderBy('total_call_minutes', ascending=False).show()

+---------+------------------+
|     city|total_call_minutes|
+---------+------------------+
|    Delhi|              6600|
|  Chennai|              4600|
|Hyderabad|              4000|
|Bangalore|              3450|
|    Kochi|              1600|
|     Pune|               500|
|   Mumbai|               250|
+---------+------------------+



In [48]:
complete_df.groupBy('state').agg(
      F.sum('sms_count').alias('total_sms')
      ).orderBy('total_sms', ascending=False).show()

+-----------+---------+
|      state|total_sms|
+-----------+---------+
|      Delhi|      910|
| Tamil Nadu|      620|
|  Telangana|      520|
|  Karnataka|      430|
|     Kerala|      250|
|Maharashtra|      100|
+-----------+---------+



In [49]:
total_rev = payments_clean.filter(F.col('payment_status') == 'Success') \
          .agg(F.sum('amount_paid').alias('total_successful_revenue'))
total_rev.show()

+------------------------+
|total_successful_revenue|
+------------------------+
|                    8089|
+------------------------+



In [50]:
complete_df.filter(F.col('payment_status') == 'Success') \
    .groupBy('city').agg(F.sum('amount_paid').alias('revenue')) \
    .orderBy('revenue', ascending=False).show()

+---------+-------+
|     city|revenue|
+---------+-------+
|Bangalore|   3695|
|    Delhi|   3197|
|  Chennai|   1996|
|Hyderabad|   1996|
|    Kochi|   1199|
|     Pune|    799|
+---------+-------+



In [51]:
complete_df.filter(F.col('payment_status') == 'Success') \
    .groupBy('plan_id', 'plan_name').agg(F.sum('amount_paid').alias('revenue')) \
    .orderBy('revenue', ascending=False).show()

+-------+-----------+-------+
|plan_id|  plan_name|revenue|
+-------+-----------+-------+
|   P102| Smart Plus|   4794|
|   P101|Smart Basic|   4491|
|   P104|Premium Max|   3597|
|UNKNOWN|       NULL|      0|
+-------+-----------+-------+



In [52]:
payments_clean.filter(F.col('payment_status') == 'Success') \
    .groupBy('payment_mode').agg(F.sum('amount_paid').alias('revenue')) \
    .orderBy('revenue', ascending=False).show()

+------------+-------+
|payment_mode|revenue|
+------------+-------+
|         UPI|   4393|
|        Card|   3696|
+------------+-------+



In [53]:
complete_df.filter(F.col('payment_status') == 'Success') \
    .groupBy('plan_id', 'plan_name').agg(F.sum('amount_paid').alias('revenue')) \
     .orderBy('revenue', ascending=False).limit(1).show()

+-------+----------+-------+
|plan_id| plan_name|revenue|
+-------+----------+-------+
|   P102|Smart Plus|   4794|
+-------+----------+-------+



In [54]:
complete_df.filter(F.col('payment_status') == 'Success') \
    .groupBy('city').agg(F.sum('amount_paid').alias('revenue')) \
    .orderBy('revenue', ascending=False).limit(1).show()

+---------+-------+
|     city|revenue|
+---------+-------+
|Bangalore|   3695|
+---------+-------+



In [55]:
w_data   = Window.orderBy(F.desc('data_used_gb'))
w_amount = Window.orderBy(F.desc('amount_paid'))
w_city   = Window.partitionBy('city').orderBy(F.desc('data_used_gb'))
w_plan   = Window.partitionBy('plan_id').orderBy(F.desc('data_used_gb'))
w_month  = Window.partitionBy('usage_month').orderBy('usage_month') \
                 .rowsBetween(Window.unboundedPreceding, Window.currentRow)
w_cust   = Window.partitionBy('customer_id').orderBy('usage_month')

In [56]:
complete_df.withColumn('data_rank', F.rank().over(w_data)) \
    .select('customer_id', 'customer_name', 'usage_month', 'data_used_gb', 'data_rank') \
     .orderBy('data_rank').show()

+-----------+-------------+-------------------+------------+---------+
|customer_id|customer_name|        usage_month|data_used_gb|data_rank|
+-----------+-------------+-------------------+------------+---------+
|        108|   Meera Nair|2026-01-01 00:00:00|          80|        1|
|        105|   Farhan Ali|2026-01-01 00:00:00|          75|        2|
|        105|   Farhan Ali|2026-01-01 00:00:00|          75|        2|
|        104|  Sneha Patel|2026-02-01 00:00:00|          58|        4|
|        104|  Sneha Patel|2026-02-01 00:00:00|          58|        4|
|        104|  Sneha Patel|2026-01-01 00:00:00|          55|        6|
|        104|  Sneha Patel|2026-01-01 00:00:00|          55|        6|
|        101| Rahul Sharma|2026-02-01 00:00:00|          50|        8|
|        101| Rahul Sharma|2026-02-01 00:00:00|          50|        8|
|        109|    Kiran Rao|2026-01-01 00:00:00|          48|       10|
|        101| Rahul Sharma|2026-01-01 00:00:00|          45|       11|
|     

In [57]:
complete_df.withColumn('payment_rank', F.rank().over(w_amount)) \
    .select('customer_id', 'customer_name', 'amount_paid', 'payment_rank') \
    .orderBy('payment_rank').show()

+-----------+-------------+-----------+------------+
|customer_id|customer_name|amount_paid|payment_rank|
+-----------+-------------+-----------+------------+
|        105|   Farhan Ali|       1199|           1|
|        105|   Farhan Ali|       1199|           1|
|        105|   Farhan Ali|       1199|           1|
|        105|   Farhan Ali|       1199|           1|
|        108|   Meera Nair|       1199|           1|
|        102|  Priya Reddy|        799|           6|
|        102|  Priya Reddy|        799|           6|
|        102|  Priya Reddy|        799|           6|
|        102|  Priya Reddy|        799|           6|
|        106|   Neha Singh|        799|           6|
|        110|  Nisha Reddy|        799|           6|
|        101| Rahul Sharma|        499|          12|
|        101| Rahul Sharma|        499|          12|
|        101| Rahul Sharma|        499|          12|
|        104|  Sneha Patel|        499|          12|
|        104|  Sneha Patel|        499|       

In [58]:

complete_df.withColumn('data_rank', F.dense_rank().over(w_data)) \
    .filter(F.col('data_rank') <= 3) \
    .select('customer_id', 'customer_name', 'usage_month', 'data_used_gb', 'data_rank') \
    .orderBy('data_rank').show()

+-----------+-------------+-------------------+------------+---------+
|customer_id|customer_name|        usage_month|data_used_gb|data_rank|
+-----------+-------------+-------------------+------------+---------+
|        108|   Meera Nair|2026-01-01 00:00:00|          80|        1|
|        105|   Farhan Ali|2026-01-01 00:00:00|          75|        2|
|        105|   Farhan Ali|2026-01-01 00:00:00|          75|        2|
|        104|  Sneha Patel|2026-02-01 00:00:00|          58|        3|
|        104|  Sneha Patel|2026-02-01 00:00:00|          58|        3|
+-----------+-------------+-------------------+------------+---------+



In [59]:
complete_df.withColumn('payment_rank', F.dense_rank().over(w_amount)) \
    .filter(F.col('payment_rank') <= 3) \
    .select('customer_id', 'customer_name', 'amount_paid', 'payment_rank') \
     .orderBy('payment_rank').show()

+-----------+-------------+-----------+------------+
|customer_id|customer_name|amount_paid|payment_rank|
+-----------+-------------+-----------+------------+
|        105|   Farhan Ali|       1199|           1|
|        105|   Farhan Ali|       1199|           1|
|        105|   Farhan Ali|       1199|           1|
|        105|   Farhan Ali|       1199|           1|
|        108|   Meera Nair|       1199|           1|
|        102|  Priya Reddy|        799|           2|
|        102|  Priya Reddy|        799|           2|
|        102|  Priya Reddy|        799|           2|
|        102|  Priya Reddy|        799|           2|
|        106|   Neha Singh|        799|           2|
|        110|  Nisha Reddy|        799|           2|
|        101| Rahul Sharma|        499|           3|
|        101| Rahul Sharma|        499|           3|
|        101| Rahul Sharma|        499|           3|
|        104|  Sneha Patel|        499|           3|
|        104|  Sneha Patel|        499|       

In [60]:

complete_df.withColumn('city_rank', F.rank().over(w_city)) \
    .filter(F.col('city_rank') == 1) \
    .select('city', 'customer_id', 'customer_name', 'data_used_gb', 'city_rank') \
    .show()

+---------+-----------+-------------+------------+---------+
|     city|customer_id|customer_name|data_used_gb|city_rank|
+---------+-----------+-------------+------------+---------+
|Bangalore|        109|    Kiran Rao|          48|        1|
|  Chennai|        104|  Sneha Patel|          58|        1|
|  Chennai|        104|  Sneha Patel|          58|        1|
|    Delhi|        105|   Farhan Ali|          75|        1|
|    Delhi|        105|   Farhan Ali|          75|        1|
|Hyderabad|        101| Rahul Sharma|          50|        1|
|Hyderabad|        101| Rahul Sharma|          50|        1|
|    Kochi|        108|   Meera Nair|          80|        1|
|   Mumbai|        103|   Amit Kumar|          12|        1|
|     Pune|        106|   Neha Singh|          28|        1|
+---------+-----------+-------------+------------+---------+



In [61]:
complete_df.withColumn('plan_rank', F.rank().over(w_plan)) \
    .filter(F.col('plan_rank') == 1) \
    .select('plan_id', 'plan_name', 'customer_id', 'customer_name', 'data_used_gb', 'plan_rank') \
    .show()

+-------+------------+-----------+-------------+------------+---------+
|plan_id|   plan_name|customer_id|customer_name|data_used_gb|plan_rank|
+-------+------------+-----------+-------------+------------+---------+
|   P101| Smart Basic|        104|  Sneha Patel|          58|        1|
|   P101| Smart Basic|        104|  Sneha Patel|          58|        1|
|   P102|  Smart Plus|        102|  Priya Reddy|          34|        1|
|   P102|  Smart Plus|        102|  Priya Reddy|          34|        1|
|   P103|Budget Saver|        103|   Amit Kumar|          12|        1|
|   P104| Premium Max|        108|   Meera Nair|          80|        1|
|   P105|        NULL|        111|   Ravi Kumar|        NULL|        1|
|UNKNOWN|        NULL|        112|  Ayesha Khan|        NULL|        1|
+-------+------------+-----------+-------------+------------+---------+



In [62]:
payments_clean.filter(F.col('payment_status') == 'Success').withColumn(
    'running_revenue',
     F.sum('amount_paid').over(
      Window.orderBy('bill_month').rowsBetween(Window.unboundedPreceding, Window.currentRow))) .select('payment_id', 'customer_id', 'bill_month', 'amount_paid', 'running_revenue').show()

+----------+-----------+-------------------+-----------+---------------+
|payment_id|customer_id|         bill_month|amount_paid|running_revenue|
+----------+-----------+-------------------+-----------+---------------+
|      5001|        101|2026-01-01 00:00:00|        499|            499|
|      5002|        102|2026-01-01 00:00:00|        799|           1298|
|      5004|        104|2026-01-01 00:00:00|        499|           1797|
|      5005|        105|2026-01-01 00:00:00|       1199|           2996|
|      5006|        106|2026-01-01 00:00:00|        799|           3795|
|      5008|        108|2026-01-01 00:00:00|       1199|           4994|
|      5009|        109|2026-01-01 00:00:00|        499|           5493|
|      5010|        110|2026-01-01 00:00:00|        799|           6292|
|      5011|        112|2026-01-01 00:00:00|          0|           6292|
|      5012|        101|2026-02-01 00:00:00|        499|           6791|
|      5013|        102|2026-02-01 00:00:00|       

In [63]:
usage_clean.withColumn('prev_month_usage', F.lag('data_used_gb', 1).over(w_cust)) .select('customer_id', 'usage_month', 'data_used_gb', 'prev_month_usage').orderBy('customer_id', 'usage_month').show()

+-----------+-------------------+------------+----------------+
|customer_id|        usage_month|data_used_gb|prev_month_usage|
+-----------+-------------------+------------+----------------+
|        101|2026-01-01 00:00:00|          45|            NULL|
|        101|2026-02-01 00:00:00|          50|              45|
|        102|2026-01-01 00:00:00|          30|            NULL|
|        102|2026-02-01 00:00:00|          34|              30|
|        103|2026-01-01 00:00:00|          12|            NULL|
|        104|2026-01-01 00:00:00|          55|            NULL|
|        104|2026-02-01 00:00:00|          58|              55|
|        105|2026-01-01 00:00:00|          75|            NULL|
|        105|2026-02-01 00:00:00|           0|              75|
|        106|2026-01-01 00:00:00|          28|            NULL|
|        107|2026-01-01 00:00:00|          10|            NULL|
|        108|2026-01-01 00:00:00|          80|            NULL|
|        109|2026-01-01 00:00:00|       

In [64]:
usage_clean.withColumn('next_month_usage', F.lead('data_used_gb', 1).over(w_cust)) \
.select('customer_id', 'usage_month', 'data_used_gb', 'next_month_usage') \
.orderBy('customer_id', 'usage_month').show()

+-----------+-------------------+------------+----------------+
|customer_id|        usage_month|data_used_gb|next_month_usage|
+-----------+-------------------+------------+----------------+
|        101|2026-01-01 00:00:00|          45|              50|
|        101|2026-02-01 00:00:00|          50|            NULL|
|        102|2026-01-01 00:00:00|          30|              34|
|        102|2026-02-01 00:00:00|          34|            NULL|
|        103|2026-01-01 00:00:00|          12|            NULL|
|        104|2026-01-01 00:00:00|          55|              58|
|        104|2026-02-01 00:00:00|          58|            NULL|
|        105|2026-01-01 00:00:00|          75|               0|
|        105|2026-02-01 00:00:00|           0|            NULL|
|        106|2026-01-01 00:00:00|          28|            NULL|
|        107|2026-01-01 00:00:00|          10|            NULL|
|        108|2026-01-01 00:00:00|          80|            NULL|
|        109|2026-01-01 00:00:00|       

In [65]:
usage_with_prev = usage_clean.withColumn('prev_month_usage', F.lag('data_used_gb', 1).over(w_cust))
usage_with_prev.filter(
    (F.col('prev_month_usage').isNotNull()) &
    (F.col('data_used_gb') > F.col('prev_month_usage'))).select('customer_id', 'usage_month', 'prev_month_usage', 'data_used_gb').orderBy('customer_id').show()

+-----------+-------------------+----------------+------------+
|customer_id|        usage_month|prev_month_usage|data_used_gb|
+-----------+-------------------+----------------+------------+
|        101|2026-02-01 00:00:00|              45|          50|
|        102|2026-02-01 00:00:00|              30|          34|
|        104|2026-02-01 00:00:00|              55|          58|
+-----------+-------------------+----------------+------------+



In [66]:
customers_clean.createOrReplaceTempView('customers')
usage_clean.createOrReplaceTempView('usage')
payments_clean.createOrReplaceTempView('payments')
plans_flat.createOrReplaceTempView('plans')
complete_df.createOrReplaceTempView('complete')

In [67]:
spark.sql("SELECT * FROM customers WHERE status = 'Active'").show()

+-----------+-------------+---------+-----------+---+------+-------+------+-------------------+
|customer_id|customer_name|     city|      state|age|gender|plan_id|status|data_quality_status|
+-----------+-------------+---------+-----------+---+------+-------+------+-------------------+
|        101| Rahul Sharma|Hyderabad|  Telangana| 35|  Male|   P101|Active|           Complete|
|        102|  Priya Reddy|Bangalore|  Karnataka| 29|Female|   P102|Active|           Complete|
|        104|  Sneha Patel|  Chennai| Tamil Nadu| 31|Female|   P101|Active|           Complete|
|        105|   Farhan Ali|    Delhi|      Delhi| 55|  Male|   P104|Active|           Complete|
|        106|   Neha Singh|     Pune|Maharashtra| 38|Female|   P102|Active|           Complete|
|        108|   Meera Nair|    Kochi|     Kerala| 48|Female|   P104|Active|           Complete|
|        109|    Kiran Rao|Bangalore|  Karnataka| 33|  Male|   P101|Active|           Complete|
|        110|  Nisha Reddy|    Delhi|   

In [68]:
spark.sql('''
    SELECT city, COUNT(*) AS customer_count
    FROM customers
    GROUP BY city
    ORDER BY customer_count DESC''').show()

+---------+--------------+
|     city|customer_count|
+---------+--------------+
|Hyderabad|             3|
|Bangalore|             2|
|   Mumbai|             2|
|    Delhi|             2|
|    Kochi|             1|
|  Chennai|             1|
|     Pune|             1|
+---------+--------------+



In [69]:
spark.sql('''
    SELECT c.plan_id, p.plan_name, SUM(py.amount_paid) AS revenue
    FROM customers c
    JOIN payments py ON c.customer_id = py.customer_id
    LEFT JOIN plans p ON c.plan_id = p.plan_id
    WHERE py.payment_status = 'Success'
    GROUP BY c.plan_id, p.plan_name
    ORDER BY revenue DESC''').show()

+-------+-----------+-------+
|plan_id|  plan_name|revenue|
+-------+-----------+-------+
|   P102| Smart Plus|   3196|
|   P101|Smart Basic|   2495|
|   P104|Premium Max|   2398|
|UNKNOWN|       NULL|      0|
+-------+-----------+-------+



In [70]:
spark.sql("SELECT * FROM complete WHERE usage_category = 'Heavy User'").show()

+-------+-----------+-------------+-----+------+---+------+------+-------------------+--------+-------------------+------------+------------+---------+-------------------+----------+-------------------+-----------+------------+--------------+-------------------+-----------+-----------+-------------+---------------+------------+-------------+--------------+----------------+----------+-------------+---------------+
|plan_id|customer_id|customer_name| city| state|age|gender|status|data_quality_status|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|data_quality_status|payment_id|         bill_month|amount_paid|payment_mode|payment_status|data_quality_status|  plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|usage_category|payment_category|churn_risk|over_usage_gb|over_usage_flag|
+-------+-----------+-------------+-----+------+---+------+------+-------------------+--------+-------------------+------------+------------+---------+---------------

In [71]:
spark.sql("SELECT customer_id, customer_name, city, status, payment_status, churn_risk FROM complete WHERE churn_risk = 'High Risk'").show()

+-----------+-------------+---------+--------+--------------+----------+
|customer_id|customer_name|     city|  status|payment_status|churn_risk|
+-----------+-------------+---------+--------+--------------+----------+
|        103|   Amit Kumar|   Mumbai|Inactive|        Failed| High Risk|
|        105|   Farhan Ali|    Delhi|  Active|       Pending| High Risk|
|        105|   Farhan Ali|    Delhi|  Active|       Pending| High Risk|
|        107|  Arjun Verma|Hyderabad|Inactive|       Pending| High Risk|
+-----------+-------------+---------+--------+--------------+----------+



In [72]:
spark.sql('''
    SELECT customer_id, customer_name, plan_id
    FROM customers
    WHERE plan_id = 'UNKNOWN'
  OR plan_id NOT IN (SELECT plan_id FROM plans)''').show()

+-----------+-------------+-------+
|customer_id|customer_name|plan_id|
+-----------+-------------+-------+
|        111|   Ravi Kumar|   P105|
|        112|  Ayesha Khan|UNKNOWN|
+-----------+-------------+-------+



In [73]:
spark.sql("""
    SELECT * FROM payments
    WHERE payment_status IN ('Failed', 'Pending')
    """).show()

+----------+-----------+-------------------+-----------+------------+--------------+-------------------+
|payment_id|customer_id|         bill_month|amount_paid|payment_mode|payment_status|data_quality_status|
+----------+-----------+-------------------+-----------+------------+--------------+-------------------+
|      5003|        103|2026-01-01 00:00:00|        299|        Cash|        Failed|           Complete|
|      5007|        107|2026-01-01 00:00:00|        299|        Cash|       Pending|           Complete|
|      5015|        105|2026-02-01 00:00:00|       1199|Not Provided|       Pending|         Incomplete|
+----------+-----------+-------------------+-----------+------------+--------------+-------------------+



In [74]:
spark.sql('''
    SELECT u.customer_id, c.customer_name, u.usage_month, u.data_used_gb
    FROM usage u
    LEFT JOIN customers c ON u.customer_id = c.customer_id
    ORDER BY u.data_used_gb DESC
    LIMIT 5''').show()

+-----------+-------------+-------------------+------------+
|customer_id|customer_name|        usage_month|data_used_gb|
+-----------+-------------+-------------------+------------+
|        108|   Meera Nair|2026-01-01 00:00:00|          80|
|        105|   Farhan Ali|2026-01-01 00:00:00|          75|
|        120|         NULL|2026-01-01 00:00:00|          60|
|        104|  Sneha Patel|2026-02-01 00:00:00|          58|
|        104|  Sneha Patel|2026-01-01 00:00:00|          55|
+-----------+-------------+-------------------+------------+



In [75]:
spark.sql('''
    SELECT payment_mode, SUM(amount_paid) AS revenue
    FROM payments
    WHERE payment_status = 'Success'
    GROUP BY payment_mode
    ORDER BY revenue DESC''').show()

+------------+-------+
|payment_mode|revenue|
+------------+-------+
|         UPI|   4393|
|        Card|   3696|
+------------+-------+



In [78]:
customers_fix = customers_clean.withColumnRenamed(
      "data_quality_status",
          "customer_dq_status"
          )

usage_fix = usage_clean.withColumnRenamed(
"data_quality_status",
"usage_dq_status")
payments_fix = payments_clean.withColumnRenamed(
"data_quality_status",
"payment_dq_status"
)


In [79]:
complete_df = (
      customers_fix
          .join(usage_fix, "customer_id", "left")
              .join(payments_fix, "customer_id", "left")
              )


In [80]:
complete_df.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- plan_id: string (nullable = false)
 |-- status: string (nullable = true)
 |-- customer_dq_status: string (nullable = false)
 |-- usage_id: integer (nullable = true)
 |-- usage_month: timestamp (nullable = true)
 |-- data_used_gb: integer (nullable = true)
 |-- call_minutes: integer (nullable = true)
 |-- sms_count: integer (nullable = true)
 |-- usage_dq_status: string (nullable = true)
 |-- payment_id: integer (nullable = true)
 |-- bill_month: timestamp (nullable = true)
 |-- amount_paid: integer (nullable = true)
 |-- payment_mode: string (nullable = true)
 |-- payment_status: string (nullable = true)
 |-- payment_dq_status: string (nullable = true)



In [81]:
complete_df.write.mode('overwrite').parquet('gold/complete')

In [85]:
complete_df.write.mode('overwrite').partitionBy('usage_month').parquet('gold/complete_partitioned')

In [89]:
incremental_data = [
      (1016, 101, '2026-03', 55.0, 1100, 140),
      (1017, 102, '2026-03', 40.0, 700, 95),
      (1018, 104, '2026-03', 62.0, 1250, 170),
      (1019, 105, '2026-03', 85.0, 1600, 220),
      (1020, 108, '2026-03', 78.0, 1500, 260)]

schema = StructType([
      StructField('usage_id', IntegerType(), True),
      StructField('customer_id', IntegerType(), True),
      StructField('usage_month', StringType(), True),
      StructField('data_used_gb', DoubleType(), True),
      StructField('call_minutes', IntegerType(), True),
      StructField('sms_count', IntegerType(), True)
])

incremental_df = spark.createDataFrame(
    incremental_data,
    schema
)

incremental_df = incremental_df.withColumn(
    'data_quality_status',
     F.lit('Complete')
)

incremental_df.show()

+--------+-----------+-----------+------------+------------+---------+-------------------+
|usage_id|customer_id|usage_month|data_used_gb|call_minutes|sms_count|data_quality_status|
+--------+-----------+-----------+------------+------------+---------+-------------------+
|    1016|        101|    2026-03|        55.0|        1100|      140|           Complete|
|    1017|        102|    2026-03|        40.0|         700|       95|           Complete|
|    1018|        104|    2026-03|        62.0|        1250|      170|           Complete|
|    1019|        105|    2026-03|        85.0|        1600|      220|           Complete|
|    1020|        108|    2026-03|        78.0|        1500|      260|           Complete|
+--------+-----------+-----------+------------+------------+---------+-------------------+



In [90]:
incremental_df.count()
incremental_df.show()

+--------+-----------+-----------+------------+------------+---------+-------------------+
|usage_id|customer_id|usage_month|data_used_gb|call_minutes|sms_count|data_quality_status|
+--------+-----------+-----------+------------+------------+---------+-------------------+
|    1016|        101|    2026-03|        55.0|        1100|      140|           Complete|
|    1017|        102|    2026-03|        40.0|         700|       95|           Complete|
|    1018|        104|    2026-03|        62.0|        1250|      170|           Complete|
|    1019|        105|    2026-03|        85.0|        1600|      220|           Complete|
|    1020|        108|    2026-03|        78.0|        1500|      260|           Complete|
+--------+-----------+-----------+------------+------------+---------+-------------------+



In [98]:
import shutil
import os

if os.path.exists("silver/usage"):
    shutil.rmtree("silver/usage")

In [99]:
from pyspark.sql.functions import col, to_timestamp

incremental_df = (incremental_df.withColumn(
      "usage_month",
       to_timestamp(col("usage_month"), "yyyy-MM"))
       .withColumn(
        "data_used_gb",
          col("data_used_gb").cast("int")))



In [101]:
usage_clean.write.mode("overwrite").parquet("silver/usage")
incremental_df.write.mode("append").parquet("silver/usage")

In [102]:
usage_updated = spark.read.parquet("silver/usage")
print("Count =", usage_updated.count())
usage_updated.orderBy("usage_id").show()

Count = 20
+--------+-----------+-------------------+------------+------------+---------+-------------------+
|usage_id|customer_id|        usage_month|data_used_gb|call_minutes|sms_count|data_quality_status|
+--------+-----------+-------------------+------------+------------+---------+-------------------+
|    1001|        101|2026-01-01 00:00:00|          45|         900|      120|           Complete|
|    1002|        102|2026-01-01 00:00:00|          30|         600|       80|           Complete|
|    1003|        103|2026-01-01 00:00:00|          12|         250|       40|           Complete|
|    1004|        104|2026-01-01 00:00:00|          55|        1100|      150|           Complete|
|    1005|        105|2026-01-01 00:00:00|          75|        1500|      200|           Complete|
|    1006|        106|2026-01-01 00:00:00|          28|         500|       60|           Complete|
|    1007|        107|2026-01-01 00:00:00|          10|         200|       20|           Complete|

In [103]:
from pyspark.sql import functions as F

customer_usage_summary = (
    usage_updated
    .groupBy("customer_id")
    .agg(
          F.sum("data_used_gb").alias("total_data_used"),
          F.sum("call_minutes").alias("total_call_minutes"),
          F.sum("sms_count").alias("total_sms_count")))
customer_usage_summary.show()

+-----------+---------------+------------------+---------------+
|customer_id|total_data_used|total_call_minutes|total_sms_count|
+-----------+---------------+------------------+---------------+
|        108|            158|              3100|            510|
|        101|            150|              3000|            390|
|        103|             12|               250|             40|
|        120|             60|              1300|            140|
|        107|             10|               200|             20|
|        102|            104|              1950|            260|
|        109|             48|               950|            100|
|        105|            160|              4550|            630|
|        110|             32|               700|             90|
|        106|             28|               500|             60|
|        104|            175|              3550|            480|
+-----------+---------------+------------------+---------------+



In [104]:
complete_updated = (
      customers_fix
      .join(usage_updated, "customer_id", "left")
       .join(payments_fix, "customer_id", "left"))

complete_updated.write \
      .mode("overwrite") \
      .partitionBy("usage_month") \
      .parquet("gold/complete_partitioned")

print("Gold output recreated successfully")

Gold output recreated successfully


In [105]:
before_count = usage_clean.count()
after_count = usage_updated.count()
print("Before Incremental Load :", before_count)
print("After Incremental Load  :", after_count)
print("New Records Added       :", after_count - before_count)

Before Incremental Load : 15
After Incremental Load  : 20
New Records Added       : 5


In [112]:
complete_updated = (
      customers_fix
          .join(plans_flat, "plan_id", "left")
           .join(usage_updated, "customer_id", "left")
            .join(payments_fix, "customer_id", "left"))
complete_updated.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- plan_id: string (nullable = false)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- status: string (nullable = true)
 |-- customer_dq_status: string (nullable = false)
 |-- plan_name: string (nullable = true)
 |-- monthly_fee: long (nullable = true)
 |-- data_limit_gb: long (nullable = true)
 |-- unlimited_calls: boolean (nullable = true)
 |-- ott_included: boolean (nullable = true)
 |-- roaming: string (nullable = true)
 |-- usage_id: integer (nullable = true)
 |-- usage_month: timestamp (nullable = true)
 |-- data_used_gb: integer (nullable = true)
 |-- call_minutes: integer (nullable = true)
 |-- sms_count: integer (nullable = true)
 |-- data_quality_status: string (nullable = true)
 |-- payment_id: integer (nullable = true)
 |-- bill_month: timestamp (nullable = true)
 |-- amount_pai

In [113]:
from pyspark.sql.functions import when, col

complete_updated = (
    complete_updated
    .withColumn("over_usage_gb",col("data_used_gb") - col("data_limit_gb"))
    .withColumn("over_usage_flag",when(col("over_usage_gb") > 0, "Yes").otherwise("No"))
    .withColumn("churn_risk",when((col("status") == "Inactive") |(col("payment_status") != "Success"),"High Risk")
    .when(col("data_used_gb") < 15, "Medium Risk").otherwise("Low Risk")))



In [114]:
customer_usage_report = (
      complete_updated
      .select(
          "customer_id",
          "customer_name",
          "city",
          "plan_name",
          "usage_month",
          "data_used_gb",
          "data_limit_gb",
          "over_usage_flag",
          "amount_paid",
          "payment_status",
          "churn_risk"))
customer_usage_report.show(truncate=False)

+-----------+-------------+---------+------------+-------------------+------------+-------------+---------------+-----------+--------------+----------+
|customer_id|customer_name|city     |plan_name   |usage_month        |data_used_gb|data_limit_gb|over_usage_flag|amount_paid|payment_status|churn_risk|
+-----------+-------------+---------+------------+-------------------+------------+-------------+---------------+-----------+--------------+----------+
|101        |Rahul Sharma |Hyderabad|Smart Basic |2026-03-01 00:00:00|55          |50           |Yes            |499        |Success       |Low Risk  |
|101        |Rahul Sharma |Hyderabad|Smart Basic |2026-03-01 00:00:00|55          |50           |Yes            |499        |Success       |Low Risk  |
|101        |Rahul Sharma |Hyderabad|Smart Basic |2026-02-01 00:00:00|50          |50           |No             |499        |Success       |Low Risk  |
|101        |Rahul Sharma |Hyderabad|Smart Basic |2026-02-01 00:00:00|50          |50   

In [115]:
plan_performance_report = (
      complete_updated
          .groupBy("plan_name")
          .agg(
          F.countDistinct("customer_id").alias("total_customers"),
          F.sum("data_used_gb").alias("total_data_usage"),
          F.avg("data_used_gb").alias("average_data_usage"),
          F.sum("amount_paid").alias("total_revenue")))
plan_performance_report.show()

+------------+---------------+----------------+------------------+-------------+
|   plan_name|total_customers|total_data_usage|average_data_usage|total_revenue|
+------------+---------------+----------------+------------------+-------------+
|        NULL|              2|            NULL|              NULL|            0|
| Smart Basic|              3|             698| 53.69230769230769|         6487|
|Budget Saver|              2|              22|              11.0|          598|
| Premium Max|              2|             478|             59.75|         9592|
|  Smart Plus|              3|             268|              33.5|         6392|
+------------+---------------+----------------+------------------+-------------+



In [116]:
city_revenue_report = (
      complete_updated
      .groupBy("city")
      .agg(
      F.countDistinct("customer_id").alias("total_customers"),
      F.sum("amount_paid").alias("total_revenue"),
      F.avg("amount_paid").alias("average_payment")))
city_revenue_report.show()

+---------+---------------+-------------+-----------------+
|     city|total_customers|total_revenue|  average_payment|
+---------+---------------+-------------+-----------------+
|Bangalore|              2|         5293|756.1428571428571|
|    Kochi|              1|         2398|           1199.0|
|  Chennai|              1|         2994|            499.0|
|   Mumbai|              2|          299|            299.0|
|     Pune|              1|          799|            799.0|
|    Delhi|              2|         7993|1141.857142857143|
|Hyderabad|              3|         3293|          411.625|
+---------+---------------+-------------+-----------------+



In [118]:
over_usage_report = (
      complete_updated
      .select(
          "customer_id",
          "customer_name",
          "plan_name",
          "data_used_gb",
          "data_limit_gb",
          "over_usage_gb"))
over_usage_report.show(truncate=False)

+-----------+-------------+------------+------------+-------------+-------------+
|customer_id|customer_name|plan_name   |data_used_gb|data_limit_gb|over_usage_gb|
+-----------+-------------+------------+------------+-------------+-------------+
|101        |Rahul Sharma |Smart Basic |55          |50           |5            |
|101        |Rahul Sharma |Smart Basic |55          |50           |5            |
|101        |Rahul Sharma |Smart Basic |50          |50           |0            |
|101        |Rahul Sharma |Smart Basic |50          |50           |0            |
|101        |Rahul Sharma |Smart Basic |45          |50           |-5           |
|101        |Rahul Sharma |Smart Basic |45          |50           |-5           |
|102        |Priya Reddy  |Smart Plus  |40          |75           |-35          |
|102        |Priya Reddy  |Smart Plus  |40          |75           |-35          |
|102        |Priya Reddy  |Smart Plus  |34          |75           |-41          |
|102        |Pri